# 대본 분석 보고서

발표 대본(슬라이드별 텍스트)을 읽어 **슬라이드별 평가 기준(Evaluation Rubric)** 을 만드는 단계의 결과를 모아 보여 줍니다.
평가 기준은 STT 평가가 "발표자가 무엇을 말했어야 하는가"의 기준으로 읽어 가는 값입니다.

이 노트북은 **보기만 합니다.** 계산은 `src/coverage_lab` 과 `src/script_coverage` 의 함수가 하고, 이 노트북은 그 함수를 불러
로컬 DB(`outputs/rubrics.sqlite`)와 실험 결과(`reports/results/*.json`)를 표로 보여 줄 뿐입니다. LLM 을 부르지 않습니다.
표를 새로 만들려면 `experiments/` 의 스크립트를 번호 순서대로 먼저 실행합니다
(`01_build_rubrics.py` → `02_reanalyze_edit.py` → `03_rubric_consistency.py`).

### 평가 기준에 들어 있는 것

| 항목 | 뜻 |
|---|---|
| 문장 역할 | 대본 문장마다 `claim`(핵심 주장) / `evidence`(근거) / `detail`(부연) / `skip`(인사·전환) |
| Core Claim | 슬라이드의 핵심 주장 한 줄 |
| Key Point | 꼭 전달해야 할 내용. 근거 문장 번호와 중요도(`critical` / `high` / `normal`)를 가짐 |
| Critical Fact | 수치·날짜·이름처럼 틀리면 안 되는 값. 규칙(`rule`)이 찾거나 LLM(`llm`)이 보탬 |
| 전달 단위 | 한 문장을 말했는지 판정할 수 있는 가장 작은 뜻 조각. STT 평가가 단위마다 말했는지 판정함 |
| Keyword | TF-IDF 로 고른 슬라이드의 중요 단어 |

In [1]:
%load_ext autoreload
%autoreload 2

import json

import pandas as pd
from IPython.display import display

from coverage_lab.datasets import script_files
from coverage_lab.results import read_result, result_table
from coverage_lab.rubric_quality import fact_values, llm_quality_report
from coverage_lab.rubric_report import key_points_table, rubric_summary, show_rubric
from coverage_lab.store import connect, load_rubric, load_rubrics
from script_coverage.version import FEATURE_VERSION

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

conn = connect()
all_rubrics = {path.stem: list(load_rubrics(conn, path.stem).values()) for path in script_files()}
print("기능 버전:", FEATURE_VERSION, "/ 평가 기준을 읽은 대본:", {name: len(rs) for name, rs in all_rubrics.items()})

기능 버전: 1.0 / 평가 기준을 읽은 대본: {'가상대본1': 9, '가상대본2': 11}


## 1. 슬라이드별 요약

슬라이드마다 한 줄입니다.

| 열 | 뜻 |
|---|---|
| KP | Key Point 개수 |
| critical/high/normal | Key Point 중요도 분포 |
| facts(rule/llm) | Critical Fact 개수. 앞은 규칙이 찾은 것, 뒤는 LLM 이 보탠 이름·용어 |
| 1차 경고 · 최종 경고 | 1차 분석 결과를 코드가 검증해서 남긴 경고 수 · 최종 결론 뒤에 남은 경고 |
| 최종 결론이 바꾼 점 | 최종 결론 LLM 이 1차 분석에서 바꾼 항목 수 |
| 전달 단위 | 전달 단위 개수 |
| top keywords | TF-IDF 상위 단어 |

In [2]:
for script_name, rubrics in all_rubrics.items():
    print(f"■ {script_name}")
    display(rubric_summary(rubrics))

■ 가상대본1


,slide,KP,critical/high/normal,facts(rule/llm),1차 경고,최종 결론이 바꾼 점,전달 단위,최종 경고,top keywords
0,1,4,1/2/1,3/1,0,0,10,unit_quote_loose:S4:자리 없는 것만이 문제가 아닙니다,"빈자리, 하루, 하루 평균, 서비스"
1,2,5,1/3/1,1/0,0,0,11,,"현장, 예약, 현장 순번, 모바일"
2,3,5,1/3/1,4/0,0,0,11,,"기록, 자리, 핵심, 차례"
3,4,4,1/3/0,7/1,0,0,13,,"모델, 오차, 시험 기간, 시험"
4,5,6,1/3/2,2/0,0,0,9,,"화면, 열람실별, 한눈, 이용자 화면"
5,6,5,1/3/1,12/0,0,0,8,,"시범, 시범 운영, 운영, 비율"
6,7,3,1/1/1,7/0,1,1,6,,"대학 도서관, 대학, 초기 대상, 예약 시스템"
7,8,4,1/2/1,4/2,0,0,6,,"학습 통계, 통계, 추가, 추가 상품"
8,9,3,1/2/0,5/0,0,0,5,,"단계, 상반기, 카페, 스터디 카페"


■ 가상대본2


,slide,KP,critical/high/normal,facts(rule/llm),1차 경고,최종 결론이 바꾼 점,전달 단위,최종 경고,top keywords
0,1,5,1/2/2,2/0,1,1,5,,"동네 빵집, 동네, 빵집, 하루 생산량"
1,2,3,0/0/3,0/4,1,0,7,no_critical_key_point,"팀원, 화면 개발, 한도윤, 한도윤 팀원"
2,3,4,1/2/1,3/1,0,0,10,,"형식, 행사, 데이터, 판매"
3,4,3,1/2/0,3/1,0,0,9,,"기간, 판매량 규모, Prophet, 평가"
4,5,5,1/3/1,4/0,0,0,7,,"기존, 품목, 오차, 폐기율"
5,6,5,1/2/2,1/0,0,0,11,,"손해, 원가, 생산량, 주문"
6,7,2,1/1/0,0/1,2,1,8,,"추천, 추천 생산량, 생산량, 에이전트"
7,8,2,1/1/0,2/1,2,0,9,,"할인, 리포트, 에이전트, 할인 정보"
8,9,4,1/2/1,8/0,0,0,6,,"폐기, 요금, 비용, 효과"
9,10,4,1/1/2,2/0,1,0,8,,"생산량, 품목별 예측, 날짜, 모습"


## 2. 슬라이드 상세

한 슬라이드의 평가 기준을 펼친 모습입니다. 수치가 많은 슬라이드(가상대본1 의 6번째)와, 사람 이름을 LLM 이 key_terms 로 보완해야
하는 슬라이드(가상대본2 의 2번째)를 봅니다. Key Point 표는 근거 문장 번호(`sentences`)와 연결된 사실(`facts`)을,
사실 표는 어느 Key Point 에 연결됐는지(`key_points`)를 보여 줍니다.

In [3]:
show_rubric(all_rubrics["가상대본1"][5])

[가상대본1 / 슬라이드 6]
문장 역할: S0 detail, S1 claim, S2 evidence, S3 evidence, S4 evidence
Core Claim: 시범 운영 결과 빈자리를 찾는 데 걸린 시간이 평균 23분에서 13분으로 약 42퍼센트 줄었다.


,id,importance,content,sentences,key_terms,facts
0,KP1,normal,2025년 3월부터 8주 동안 제휴 도서관 3곳에서 시범 운영을 했다.,[0],[],"[CF1, CF2, CF3]"
1,KP2,critical,빈자리를 찾는 데 걸린 시간이 평균 23분에서 13분으로 약 42퍼센트 줄었다.,[1],[],"[CF4, CF5, CF6, CF7]"
2,KP3,high,알림을 받고 10분 안에 자리를 잡은 비율이 76%이다.,[2],[],"[CF8, CF9]"
3,KP4,high,노쇼 비율이 18%에서 7%로 낮아졌다.,[3],[],"[CF10, CF11]"
4,KP5,high,시범 운영 뒤 설문에서 이용자의 85퍼센트 이상이 계속 쓰고 싶다고 답했다.,[4],[],[CF12]


,id,type,value,normalized,qualifier,source,key_points,importance
0,CF1,date,2025년 3월,2025-03,NaN,rule,[KP1],normal
1,CF2,duration,8주 동안,8주,NaN,rule,[KP1],normal
2,CF3,quantity,3곳,3곳,NaN,rule,[KP1],normal
3,CF4,quantity,"1,860명","1,860명",NaN,rule,[KP2],critical
4,CF5,duration,평균 23분,23분,평균,rule,[KP2],critical
5,CF6,duration,13분,13분,NaN,rule,[KP2],critical
6,CF7,percentage,약 42퍼센트,42%,약,rule,[KP2],critical
7,CF8,duration,10분,10분,NaN,rule,[KP3],high
8,CF9,percentage,76%,76%,NaN,rule,[KP3],high
9,CF10,percentage,18%,18%,NaN,rule,[KP4],high


전달 단위 (STT 평가가 단위마다 말했는지 판정한다)


,id,단위,대본 표현,사실,출처
0,S0-U1,2025년 3월부터 시범 운영함,2025년 3월부터,[CF1],llm
1,S0-U2,8주 동안 시범 운영함,8주 동안,[CF2],llm
2,S0-U3,제휴 도서관 3곳에서 시범 운영함,제휴 도서관 3곳,[CF3],llm
3,S1-U1,"참여한 이용자는 1,860명이었음","참여한 이용자는 1,860명",[CF4],llm
4,S1-U2,빈자리를 찾는 데 걸린 시간이 평균 23분에서 13분으로 약 42퍼센트 줄어듦,빈자리를 찾는 데 걸린 시간은 평균 23분에서 13분으로 약 42퍼센트 줄었습니다,"[CF5, CF6, CF7]",llm
5,S2-U1,알림을 받고 10분 안에 자리를 잡은 비율은 76%였음,알림을 받고 10분 안에 자리를 잡은 비율은 76%,"[CF8, CF9]",llm
6,S3-U1,노쇼 비율이 18%에서 7%로 낮아짐,노쇼 비율도 18%에서 7%로 낮아졌습니다,"[CF10, CF11]",llm
7,S4-U1,시범 운영 뒤 설문에서 이용자의 85퍼센트 이상이 계속 쓰고 싶다고 답함,이용자의 85퍼센트 이상이 계속 쓰고 싶다고 답했습니다,[CF12],llm


Keywords: 시범(0.38), 시범 운영(0.38), 운영(0.38), 비율(0.32), 퍼센트(0.22), 동안(0.22), 동안 제휴(0.22), 설문(0.22)
1차 검증 경고: -
최종 결론이 바꾼 점: -
이름·용어 결정: -
최종 경고: -


In [4]:
show_rubric(all_rubrics["가상대본2"][1])

[가상대본2 / 슬라이드 2]
문장 역할: S0 skip, S1 detail, S2 detail, S3 detail
Core Claim: 


,id,importance,content,sentences,key_terms,facts
0,KP1,normal,한도윤 팀원은 판매량 예측 모델과 성능 검증을 맡은 팀원이다.,[1],[한도윤],[CF1]
1,KP2,normal,서민재 팀원은 서비스 기획과 대화형 에이전트를 만든 팀원이다.,[2],"[서민재, 대화형 에이전트]","[CF2, CF3]"
2,KP3,normal,"오예린 팀원은 데이터 수집과 화면 개발, 발표 자료 정리를 담당한 팀원이다.",[3],[오예린],[CF4]


,id,type,value,normalized,qualifier,source,key_points,importance
0,CF1,term,한도윤,한도윤,None,llm,[KP1],normal
1,CF2,term,서민재,서민재,None,llm,[KP2],normal
2,CF3,term,대화형 에이전트,대화형 에이전트,None,llm,[KP2],normal
3,CF4,term,오예린,오예린,None,llm,[KP3],normal


전달 단위 (STT 평가가 단위마다 말했는지 판정한다)


,id,단위,대본 표현,사실,출처
0,S1-U1,한도윤 팀원이 판매량 예측 모델을 맡음,한도윤 팀원은 판매량 예측 모델,[CF1],llm
1,S1-U2,한도윤 팀원이 성능 검증을 맡음,성능 검증,[CF1],llm
2,S2-U1,서민재 팀원이 서비스 기획을 맡음,서민재 팀원은 서비스 기획,[CF2],llm
3,S2-U2,서민재 팀원이 대화형 에이전트를 만듦,대화형 에이전트,"[CF2, CF3]",llm
4,S3-U1,오예린 팀원이 데이터 수집을 담당함,오예린 팀원은 데이터 수집,[CF4],llm
5,S3-U2,오예린 팀원이 화면 개발을 담당함,화면 개발,[CF4],llm
6,S3-U3,오예린 팀원이 발표 자료 정리를 담당함,발표 자료 정리,[CF4],llm


Keywords: 팀원(0.38), 화면 개발(0.18), 한도윤(0.18), 한도윤 팀원(0.18), 정리(0.18), 팀원별(0.18), 팀원별 역할(0.18), 서민재 팀원(0.18)
1차 검증 경고: ['no_critical_key_point']
최종 결론이 바꾼 점: -
이름·용어 결정: ['한도윤=유지(팀원 이름이므로 그대로 전달되어야 한다.)', '서민재=유지(팀원 이름이므로 그대로 전달되어야 한다.)', '대화형 에이전트=유지(서비스의 전문 기능 용어이므로 그대로 전달되어야 한다.)', '오예린=유지(팀원 이름이므로 그대로 전달되어야 한다.)']
최종 경고: ['no_critical_key_point']


## 3. 저장된 평가 기준 (JSON)

DB 에 저장된 모양 그대로입니다. STT 평가는 이 JSON 을 읽어 채점합니다 (본문 · 문장 · 키워드는 길어서 뺐습니다).
`meta.rubric_id` 는 같은 대본 · 같은 분석 설정이면 같은 값이라, 채점 결과에 함께 남겨 어떤 기준으로 채점했는지 추적합니다.

In [5]:
stored = load_rubric(conn, "가상대본1", 7)
print(json.dumps(stored.model_dump(exclude={"sentences", "normalized_script", "keywords"}), ensure_ascii=False, indent=2))

{
  "meta": {
    "rubric_id": "8d7e8f07b9d6a858",
    "script_name": "가상대본1",
    "slide_number": 7,
    "content_hash": "2c80d08e9afdcd4a1ad597ac61a127130a7532b1085ce5a0a6aa508ce684211d",
    "rubric_schema_version": "1.1",
    "llm_model": "openai/gpt-5.6-luna",
    "llm_config_hash": "f7b2ef94d4e7",
    "final_config_hash": "83cb79b10a03",
    "unit_config_hash": "1e964b21a6bf",
    "created_at": "2026-10-06T04:58:53+00:00"
  },
  "sentence_roles": [
    "evidence",
    "claim",
    "detail"
  ],
  "core_claim": "열람실 좌석 예약 시스템 시장의 초기 대상은 약 900곳이며 연간 시장 규모는 약 24억 원에서 36억 원이다.",
  "key_points": [
    {
      "id": "KP1",
      "content": "국내 공공도서관은 약 1,200곳이고 대학 도서관은 약 430곳이다.",
      "importance": "high",
      "sentence_indices": [
        0
      ],
      "source_quote": "국내 공공도서관은 약 1,200곳이고, 대학 도서관은 약 430곳입니다.",
      "source_span": [
        0,
        40
      ],
      "key_terms": [],
      "fact_ids": [
        "CF1",
        "CF2"
      ]
    },
    {
      "id": "KP2",
   

## 4. 대본을 고쳤을 때

가상 시나리오: 발표자가 가상대본2 의 5번 슬라이드에서 결과 문장 하나만 고쳐(`폐기율 약 18%` → `약 23%`, 정확도 91% 문장 추가)
`가상대본2_수정` 이라는 이름으로 다시 넣었습니다. 바뀐 슬라이드만 다시 분석하고 나머지는 캐시를 쓰므로, 두 평가 기준을 나란히 비교할 수 있습니다.
사실의 `type:normalized` 값으로 사라진 것과 새로 생긴 것을 봅니다.

In [6]:
before = load_rubric(conn, "가상대본2", 5)
after = load_rubric(conn, "가상대본2_수정", 5)
print("사라진 사실:", fact_values(before) - fact_values(after))
print("새로 생긴 사실:", fact_values(after) - fact_values(before))
key_points_table(after)

사라진 사실: {'percentage:18%'}
새로 생긴 사실: {'percentage:23%', 'percentage:91%'}


,id,importance,content,sentences,key_terms,facts
0,KP1,critical,기존 방식과 비교해 폐기율은 약 23퍼센트 줄었고 오후 품절 알림의 정확도는 91퍼센트이다.,[1],[],"[CF1, CF2]"
1,KP2,high,품목별 판매량 예측 오차는 MAPE 기준 11.4%이다.,[2],[MAPE],"[CF3, CF4]"
2,KP3,normal,새로 나온 품목은 판매 기록이 적어 예측 오차가 30% 이상으로 커진다.,[3],[],[CF5]
3,KP4,high,예측 오차가 큰 새 품목은 비슷한 기존 품목의 판매 흐름을 빌려 와 예측하도록 보완한다.,[4],[],[]
4,KP5,high,예측이 불안정한 품목은 화면에 따로 표시해 사장님이 직접 판단할 수 있게 한다.,[5],[],[]


## 5. LLM 결과 품질

LLM 이 만든 부분(문장 역할 · Key Point · 이름 결정 · 전달 단위)이 채점 기준으로 쓸 만한지 보는 지표입니다.
**LLM 응답이 바뀌면 값도 바뀌므로** 합격선이 아니라 변화를 보는 지표로 읽습니다. 모든 대본의 최종 평가 기준을 합쳐 셉니다.
예를 들어 `claim 문장이 2개 이상인 슬라이드`, `critical 이 2개 이상인 슬라이드`는 핵심 주장이 한 개로 모이지 않는 경우이고,
`대본에 없어 버린 전달 단위`는 LLM 이 대본에 없는 표현을 단위로 만들어 코드가 버린 경우입니다.

In [7]:
llm_quality_report([r for rs in all_rubrics.values() for r in rs])

,지표,값
0,Key Point 수,80
1,중요도 분포 (critical/high/normal),19/40/21
2,claim 문장이 2개 이상인 슬라이드,0
3,Key Point 에 안 들어간 문장 (skip 제외),0
4,잘못된 문장 번호 / 역할이 빠진 문장,0
5,critical 없는 슬라이드,1
6,Key Point 에 연결된 수치 사실,57/57
7,LLM 이 보탠 용어(term) 사실,12
8,대본에 없는 key_term,0
9,이름·용어 후보 유지/제외 (최종 결론),25/6


## 6. 일관성 (같은 대본을 여러 번 분석했을 때)

채점 기준은 매번 같아야 합니다. 같은 대본으로 평가 기준을 여러 벌 만들어, **1차 결과와 최종 결과가 각각 얼마나 흔들리는지** 비교한 결과입니다
(`experiments/03_rubric_consistency.py` 가 `reports/results/rubric_consistency.json` 에 남깁니다).
k 번째 벌 = k 번째 1차 분석 응답 → 1차 결과(코드) → 최종 결론 응답 → 최종 결과(코드). Key Point 는 근거 문장 집합이 절반 이상 겹치면 같은 것으로 봅니다.

| 지표 | 뜻 |
|---|---|
| 문장 역할 일치율 | 같은 문장에 같은 역할을 준 비율 |
| KP 대응률 | 0번 기준의 Key Point 가 다른 기준에도 (근거가 겹치는 형태로) 있는 비율 |
| 중요도 일치율 | 대응된 Key Point 끼리 중요도가 같은 비율 |
| critical 위치 일치 | critical 로 고른 Key Point 의 근거가 같은 비율 (둘 다 critical 이 없으면 일치) |
| 이름·용어 일치 | 평가 기준에 들어간 이름·용어 사실 집합의 자카드 유사도 |
| 점수 차이 | 같은 가상 발화(문장마다 80% 확률로 말함)를 각 기준으로 채점했을 때 점수 차이 (100점 만점) |

In [8]:
consistency = read_result("rubric_consistency")
print("모델:", consistency["model"], "/ 기능 버전:", consistency["feature_version"], "/ 벌 수:", consistency["samples"])
result_table("rubric_consistency", "summary").round(2)

모델: openai/gpt-5.6-luna / 기능 버전: 1.0 / 벌 수: 3


,문장 역할 일치율,KP 대응률,중요도 일치율,critical 위치 일치,이름·용어 일치,점수 차이 평균,점수 차이 중앙값,점수 차이 상위 10%
1차,0.83,1.0,0.81,0.80,0.92,4.15,0.0,12.5
최종,0.84,1.0,0.81,0.82,0.77,4.28,0.0,12.5


슬라이드별 최종 결과입니다. `KP 수` · `critical 수` 는 벌마다의 값을 `/` 로 이었습니다.

In [9]:
pd.DataFrame(consistency["per_slide_final"]).round(2)

,file,slide,KP 수,critical 수,문장 역할 일치율,KP 대응률,중요도 일치율,critical 위치 일치,이름·용어 일치,점수 차이 평균
0,가상대본1,1,4/4/5,1/1/1,0.71,1.0,0.38,0.0,1.0,8.72
1,가상대본1,2,5/4/4,1/1/1,1.00,1.0,1.00,1.0,0.5,4.18
2,가상대본1,3,5/5/5,1/1/1,0.60,1.0,0.60,1.0,0.5,5.77
3,가상대본1,4,4/5/4,1/1/1,1.00,1.0,1.00,1.0,1.0,3.72
4,가상대본1,5,6/5/5,1/1/1,1.00,1.0,1.00,1.0,1.0,2.56
5,가상대본1,6,5/5/5,1/1/1,0.80,1.0,0.80,0.5,1.0,3.43
6,가상대본1,7,3/3/4,1/1/2,1.00,1.0,1.00,1.0,1.0,4.96
7,가상대본1,8,4/4/3,1/1/1,1.00,1.0,0.88,1.0,0.0,3.94
8,가상대본1,9,3/3/3,1/1/1,0.67,1.0,0.33,0.0,1.0,7.74
9,가상대본2,1,5/5/5,1/1/1,0.83,1.0,0.80,1.0,1.0,2.95
